In [2]:
import json
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# Helper: unwrap nested leaf values safely
def unwrap(v):
    while isinstance(v, dict):
        v = next(iter(v.values()))
    return v


# 1. Load data from a single instance
file_path = r"Deterministic\Model Files\Segment Test\results\MVP_DE_results_T_10_delta_5_scen_1_trial_1_inv_1_cap._1_cap.inc._1.json"

with open(file_path, "r") as f:
    data = json.load(f)

# 2. Flatten Q [vaccine][producer][start][end][discount]
q_list = []
for vaccine, producers in data["Q"].items():
    for producer, starts in producers.items():
        for s, ends in starts.items():
            for e, discounts in ends.items():
                for d, val in discounts.items():
                    val = unwrap(val)
                    if val > 0:
                        q_list.append(
                            {
                                "vaccine": vaccine,
                                "producer": producer,
                                "start_time": int(s),
                                "end_time": int(e),
                                "discount_level": int(d),
                                "Q_val": val,
                            }
                        )
df_q = pd.DataFrame(q_list)

# 3. Flatten X [vaccine][producer][time]
x_list = []
for vaccine, producers in data["X"].items():
    for producer, times in producers.items():
        for t, scenarios in times.items():
            val = sum(scenarios.values())
            if val > 0:
                x_list.append(
                    {
                        "vaccine": vaccine,
                        "producer": producer,
                        "time": int(t),
                        "X_val": val,
                    }
                )
df_x = pd.DataFrame(x_list)

# 4. Relational Non-Equi Join
df_mapped = pd.merge(df_x, df_q, on=["vaccine", "producer"], how="left")
df_mapped = df_mapped[
    (df_mapped["time"] >= df_mapped["start_time"])
    & (df_mapped["time"] <= df_mapped["end_time"])
]

# 5. Group by contract to sum X values
analysis_df = (
    df_mapped.groupby(
        [
            "vaccine",
            "producer",
            "start_time",
            "end_time",
            "discount_level",
            "Q_val",
        ]
    )
    .agg(Sum_of_X=("X_val", "sum"))
    .reset_index()
)

# 6. FILTER OUT PERFECT MATCHES
# We calculate the absolute discrepancy and drop everything within our tolerance
analysis_df["Discrepancy"] = analysis_df["Sum_of_X"] - analysis_df["Q_val"]
mismatches_df = analysis_df[analysis_df["Discrepancy"].abs() >= 1e-2].copy()

# ==========================================
# PLOTTING THE MISMATCHES (IF ANY EXIST)
# ==========================================
if mismatches_df.empty:
    print(
        "🎉 Clean slate! No chart generated because 100% of the contracts matched perfectly."
    )
else:
    # Create clean labels for the mismatched entries
    mismatches_df["Contract_Label"] = (
        mismatches_df["vaccine"]
        + " ("
        + mismatches_df["producer"]
        + ")\nWindow: "
        + mismatches_df["start_time"].astype(str)
        + "-"
        + mismatches_df["end_time"].astype(str)
        + " | Lvl: "
        + mismatches_df["discount_level"].astype(str)
    )

    # Sort descending by the size of the commitment gap
    mismatches_df["Absolute_Error"] = mismatches_df["Discrepancy"].abs()
    mismatches_df = mismatches_df.sort_values(
        by="Absolute_Error", ascending=False
    ).reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(10, 5))

    x_indices = np.arange(len(mismatches_df))
    bar_width = 0.35

    # Side-by-side comparison for the broken thresholds
    ax.bar(
        x_indices - bar_width / 2,
        mismatches_df["Q_val"],
        bar_width,
        label="Contract Commitment ($Q_{val}$)",
        color="#d62728",
        alpha=0.85,
    )  # Red for target
    ax.bar(
        x_indices + bar_width / 2,
        mismatches_df["Sum_of_X"],
        bar_width,
        label="Total Deliveries ($\sum X_{val}$)",
        color="#7f7f7f",
        alpha=0.85,
    )  # Gray for execution

    # Chart Polish
    ax.set_ylabel("Quantity Volume", fontsize=11)
    ax.set_title(
        f"Mismatched Commitments Breakdown ({len(mismatches_df)} Discrepancies Found)",
        fontsize=13,
        fontweight="bold",
        pad=15,
    )
    ax.set_xticks(x_indices)
    ax.set_xticklabels(
        mismatches_df["Contract_Label"], rotation=25, ha="right", fontsize=9
    )
    ax.legend(fontsize=10)
    ax.grid(axis="y", linestyle="--", alpha=0.4)

    plt.tight_layout()

    # Save the isolated discrepancies chart
    plt.savefig("isolated_mismatches_no_W.png", dpi=300)
    print(
        f"🚨 Plot successfully generated and saved as 'isolated_mismatches.png'"
    )
    print(mismatches_df[["Contract_Label", "Q_val", "Sum_of_X", "Discrepancy"]])

🎉 Clean slate! No chart generated because 100% of the contracts matched perfectly.
